In [ ]:
import pandas as pd
import numpy as np
from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc, concordance_index_ipcw
import warnings
from tqdm import tqdm
import os
from sklearn.metrics import confusion_matrix

In [ ]:
# ---- base folder names (define once, reuse below) ----
ckd_event_full     = "365day_future_prediction_outputs_50_full_stage_filter_v8"
ckd_event_subset   = "365day_future_prediction_outputs_50_subset_100_stage_filter_v1"
ckd_patient_full   = "365day_future_prediction_outputs_50_full_stage_filter_patient_level_v2"
ckd_patient_subset = "365day_future_prediction_outputs_50_subset_1000_stage_filter_patient_level_v2"

eskd_event_full   = "365day_future_prediction_outputs_full_stage_filter_eskd_v2"
eskd_event_subset = "365day_future_prediction_outputs_subset_100_stage_filter_eskd_v2"
eskd_patient_full = "365day_future_prediction_outputs_full_stage_filter_eskd_v2_patient_level"
# eskd_event_full   = "365day_future_prediction_outputs_full_stage_filter_eskd_v6"
# eskd_patient_full = "365day_future_prediction_outputs_full_stage_filter_eskd_v6_patient_level"
eskd_event_full   = "365day_future_prediction_outputs_full_stage_filter_eskd_v7"
# eskd_patient_full = "365day_future_prediction_outputs_full_stage_filter_eskd_v6_patient_level"

# no subset patient-level eskd data yet -- add eskd_patient_subset here when it exists

mlp_event_full   = "365day_future_prediction_outputs_50_full_stage_filter_mlp_v1"
mlp_patient_full = "365day_future_prediction_outputs_50_full_stage_filter_mlp_v1_patient_level"
# new MLP variant 

# ---- per-model file lists (define once, reuse below) ----
ckd_clf_dirs = [
    "/LSTM_365DayFutureTarget_detailed_outputs.csv",
    "/MLP_365DayFutureTarget_detailed_outputs.csv",
    "/RNN_365DayFutureTarget_detailed_outputs.csv",
    "/TCN_365DayFutureTarget_detailed_outputs.csv",
    "/Transformer_365DayFutureTarget_detailed_outputs.csv",
]
ckd_surv_dirs = [
    "/DeepSurv_LSTM_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_MLP_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_RNN_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_TCN_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_Transformer_365DayFutureTarget_detailed_outputs.csv",
]
eskd_event_dirs   = ["/XGBoost_365DayFuture_Classifier_detailed_outputs_classification.csv"]
eskd_patient_dirs = ["/XGBoost_365DayFuture_Classifier_detailed_outputs_classification_pt_lvl.csv"]

mlp_clf_dirs          = ["/MLP_365DayFutureTarget_detailed_outputs.csv"]
mlp_surv_dirs         = ["/DeepSurv_MLP_365DayFutureTarget_detailed_outputs.csv"]
mlp_clf_patient_dirs  = ["/MLP_365DayFutureTarget_detailed_outputs_pt_lvl.csv"]
mlp_surv_patient_dirs = ["/DeepSurv_MLP_365DayFutureTarget_detailed_outputs_pt_lvl.csv"]

# display-name overrides for the new MLP variant -- its filenames are identical to the existing
# MLP's, so evaluate_models() (which names each model after its basename) would otherwise collide
mlp_clf_names          = ["MLP_alt"]
mlp_surv_names         = ["DeepSurv_MLP_alt"]

def paths(base, dirs):
    return [f"./{base}{d}" for d in dirs]

def named_paths(base, dirs, names):
    # like paths(), but pairs each file with an explicit display name: (name, filepath).
    # Use when merging in files whose basenames collide with something else already in the list.
    return [(names[i], f"./{base}{dirs[i]}") for i in range(len(dirs))]

# ---- naming convention: <ckd|eskd|ckd_eskd>_<clf|surv>_<event|patient>_<full|subset> ----
presets = {
    # !!
    "ckd_eskd_clf_event_full": {
        "fp": f"./{ckd_event_full}", "modifier": "classification",
        "filepaths": paths(ckd_event_full, ckd_clf_dirs) + paths(eskd_event_full, eskd_event_dirs),
    },
    "ckd_eskd_clf_event_subset": {
        "fp": f"./{ckd_event_subset}", "modifier": "classification",
        "filepaths": paths(ckd_event_subset, ckd_clf_dirs) + paths(eskd_event_subset, eskd_event_dirs),
    },
    # !!
    # "ckd_eskd_clf_patient_full": {
    #     "fp": f"./{ckd_patient_full}", "modifier": "classification",
    #     "filepaths": paths(ckd_patient_full, ckd_clf_dirs) + paths(eskd_patient_full, eskd_patient_dirs),
    # },

    "ckd_clf_event_full": {
        "fp": f"./{ckd_event_full}", "modifier": "classification",
        "filepaths": paths(ckd_event_full, ckd_clf_dirs),
    },
    "ckd_clf_event_subset": {
        "fp": f"./{ckd_event_subset}", "modifier": "classification",
        "filepaths": paths(ckd_event_subset, ckd_clf_dirs),
    },
    # "ckd_clf_patient_full": {
    #     "fp": f"./{ckd_patient_full}", "modifier": "classification",
    #     "filepaths": paths(ckd_patient_full, ckd_clf_dirs),
    # },
    # !!
    "ckd_surv_event_full": {
        "fp": f"./{ckd_event_full}", "modifier": "deepsurv",
        "filepaths": paths(ckd_event_full, ckd_surv_dirs),
    },
    "ckd_surv_event_subset": {
        "fp": f"./{ckd_event_subset}", "modifier": "deepsurv",
        "filepaths": paths(ckd_event_subset, ckd_surv_dirs),
    },
    # !!
    # "ckd_surv_patient_full": {
    #     "fp": f"./{ckd_patient_full}", "modifier": "deepsurv",
    #     "filepaths": paths(ckd_patient_full, ckd_surv_dirs),
    # },

    "eskd_clf_event_full": {
        "fp": f"./{eskd_event_full}", "modifier": "classification",
        "filepaths": paths(eskd_event_full, eskd_event_dirs),
    },
    "eskd_clf_event_subset": {
        "fp": f"./{eskd_event_subset}", "modifier": "classification",
        "filepaths": paths(eskd_event_subset, eskd_event_dirs),
    },
    # "eskd_clf_patient_full": {
    #     "fp": f"./{eskd_patient_full}", "modifier": "classification",
    #     "filepaths": paths(eskd_patient_full, eskd_patient_dirs),
    # },

    # new MLP variant -- own directory, event/full level only so far (no subset yet)
    "mlp_clf_event_full": {
        "fp": f"./{mlp_event_full}", "modifier": "classification",
        "filepaths": paths(mlp_event_full, mlp_clf_dirs),
    },
    "mlp_surv_event_full": {
        "fp": f"./{mlp_event_full}", "modifier": "deepsurv",
        "filepaths": paths(mlp_event_full, mlp_surv_dirs),
    },

    # "mlp_clf_patient_full": {
    #     "fp": f"./{mlp_patient_full}", "modifier": "classification",
    #     "filepaths": paths(mlp_patient_full, mlp_clf_patient_dirs),
    # },
    # "mlp_surv_patient_full": {
    #     "fp": f"./{mlp_patient_full}", "modifier": "deepsurv",
    #     "filepaths": paths(mlp_patient_full, mlp_surv_patient_dirs),
    # },
}


In [ ]:

#  "mlp_surv_event_full"
# preset_modifier =  "mlp_clf_event_full"
preset_modifier = "eskd_clf_event_full"
# preset_modifier = "eskd_clf_patient_full"
out_file_mod = ""
if "surv" in preset_modifier:
    out_file_mod = "DeepSurv_"
preset = presets[preset_modifier]
# ==== EDIT ^^ TO SWITCH RUNS ====

fp = preset["fp"]
modifier = preset["modifier"]
filepaths = preset["filepaths"]

print("preset: ", preset_modifier)
print("fp:      ", fp)
print("modifier:", modifier)
print("filepaths:")
for p in filepaths:
    print("  ", p)


In [ ]:
test_df = pd.read_csv(filepaths[0])

In [ ]:
test_df['cl_true_label'].unique(
)

In [ ]:
# variable name, not metadata (results)
temp_results = filepaths[0]
test_meta = pd.read_csv(temp_results)

In [ ]:
test_meta.head()

In [ ]:
test_meta = test_meta.rename(columns={'EventDate': 'date',})

In [ ]:
test_meta.head()

In [ ]:
len(test_meta['PatientID'].unique())

In [ ]:

# %%
def select_encounters(df):
    """
        selects a single encounter row for each patient

        -For patients who progress in CKD stage (CKD_stage_numeric increases),
        selects the last encounter before the patient progresses in ckd stage
        -For patients whose CKD stage never changes, selects a random encounter.
        """
    df = df.copy()
    # date_column = "EncounterDate"
    date_column = "date"

    df[date_column] = pd.to_datetime(df[date_column])
    df = df.sort_values(by=['PatientID', date_column]).reset_index(drop=True)
    df['next_stage'] = df.groupby('PatientID')['CKD_stage_numeric'].shift(-1)
    # find changes in ckd stage
    df['last_encounter'] = df['next_stage'] > df['CKD_stage_numeric']

    selected_indices = []

    for patient, group in df.groupby('PatientID'):
        progressions = group[group['last_encounter']]

        if not progressions.empty:
            # patient progresses
            last_pre_progression_index = progressions.index.max()
            selected_indices.append(last_pre_progression_index)
        else:
            # patient does not progress
            random_index = np.random.choice(group.index)
            selected_indices.append(random_index)

    result_df = df.loc[selected_indices].drop(columns=['next_stage', 'last_encounter']).reset_index(drop=True)
    return result_df

test_meta = select_encounters(test_meta)

In [ ]:
test_meta

In [ ]:
len(test_meta['PatientID'].unique())

In [ ]:
output_dir = fp + "_patient_level"
print(output_dir)

In [ ]:
try:
    os.mkdir(output_dir)
except FileExistsError:
    pass

In [ ]:
dirs = os.listdir(fp)
dirs


In [ ]:
output_dir

In [ ]:
test_file_name = dirs[0]
new_file_name = out_file_mod + dirs[0].split(".")[0] + "_pt_lvl." + dirs[0].split(".")[1] 
print(output_dir)
print(new_file_name)
new_path = os.path.join(output_dir, new_file_name)
print(new_path)


In [ ]:
test_meta.to_csv(new_path)

In [ ]:
test_meta.head()